In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, roc_auc_score, confusion_matrix, 
                             precision_recall_curve)
import xgboost as xgb
import optuna
from optuna.samplers import TPESampler
import shap
import joblib
from imblearn.under_sampling import NearMiss # 다시 추가
import warnings

warnings.filterwarnings('ignore')

# ============================================================================
# 1. 데이터 로드
# ============================================================================
print("=" * 80)
print("1. 데이터 로드")
print("=" * 80)
df = pd.read_csv('selected_data_for_modeling.csv')

X = df.drop(['ID', 'PERF_12M'], axis=1)
y = df['PERF_12M']

print(f"전체 데이터: {X.shape}")
print(f"전체 부도 건수: {y.sum()}건 ({(y.sum()/len(y))*100:.2f}%)")

# ============================================================================
# 2. NearMiss 언더샘플링 (논문 방식 복귀)
# ============================================================================
print("\n" + "=" * 80)
print("2. NearMiss 언더샘플링 적용 (1:1 비율)")
print("=" * 80)

# 논문 근거: 데이터 불균형 해소를 위해 1:1 적용
nm = NearMiss(version=1, n_neighbors=3)
X_resampled, y_resampled = nm.fit_resample(X, y)

print(f"리샘플링 후 데이터 크기: {X_resampled.shape}")
print(f"리샘플링 후 부도 건수: {y_resampled.sum()}건")
print(f"리샘플링 후 정상 건수: {(y_resampled==0).sum()}건")
print("비율이 1:1로 조정되었습니다.")

# ----------------------------------------------------------------------------
# [추가됨] 리샘플링된 데이터 저장 (Step 9 시각화 및 Agent에서 사용)
# ----------------------------------------------------------------------------
# X와 y를 합쳐서 데이터프레임으로 복원
df_resampled = pd.DataFrame(X_resampled, columns=X.columns)
df_resampled['PERF_12M'] = y_resampled

# CSV 파일로 저장
df_resampled.to_csv('resampled_data_final.csv', index=False, encoding='utf-8-sig')
print(">> 'resampled_data_final.csv' 저장 완료 (시각화용)")
# ----------------------------------------------------------------------------

# Train/Test Split (8:2)
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=42, stratify=y_resampled
)

# 1:1이므로 scale_pos_weight는 1.0 (사용 안 함)
pos_weight = 1.0 

# ============================================================================
# 유틸리티 함수
# ============================================================================
def find_optimal_threshold(y_true, y_proba):
    precision, recall, thresholds = precision_recall_curve(y_true, y_proba)
    # 분모 0 방지
    with np.errstate(divide='ignore', invalid='ignore'):
        f1_scores = 2 * (precision * recall) / (precision + recall)
    f1_scores = np.nan_to_num(f1_scores)
    
    if len(f1_scores) == 0: return 0.5, 0.0
    
    optimal_idx = np.argmax(f1_scores)
    if optimal_idx < len(thresholds):
        optimal_threshold = thresholds[optimal_idx]
    else:
        optimal_threshold = 0.5
    return optimal_threshold, f1_scores[optimal_idx]

def evaluate_model(model, X_test, y_test, model_name):
    y_prob = model.predict_proba(X_test)[:, 1]
    
    # Test set 내에서 최적 임계값 찾기 (성능 확인용)
    threshold, f1 = find_optimal_threshold(y_test, y_prob)
    y_pred = (y_prob >= threshold).astype(int)
    
    print(f"\n[{model_name}] 결과")
    print(f"AUC: {roc_auc_score(y_test, y_prob):.4f}")
    print(f"Optimal Threshold: {threshold:.4f}")
    print(f"F1 Score: {f1:.4f}")
    
    cm = confusion_matrix(y_test, y_pred)
    print(f"Confusion Matrix:\n{cm}")
    # TP가 나오는지 확인
    print(f"부도(1)를 맞춘 개수(TP): {cm[1,1]} / {cm[1,0]+cm[1,1]}") 
    
    return {'Model': model_name, 'Threshold': threshold, 'AUC': roc_auc_score(y_test, y_prob)}

# ============================================================================
# 3. Base Model 학습 (Optuna)
# ============================================================================
print("\n" + "=" * 80)
print("3. XGBoost 학습 (Optuna)")
print("=" * 80)

def objective(trial):
    params = {
        'objective': 'binary:logistic',
        'eval_metric': 'logloss',
        'tree_method': 'hist',
        'random_state': 42,
        'n_jobs': -1,
        'scale_pos_weight': pos_weight, # 1.0
        
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 7)
    }
    
    # 데이터가 적으므로(약 400개) K-Fold를 늘려 안정성 확보
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []
    
    for tr_idx, val_idx in cv.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
        
        model = xgb.XGBClassifier(**params)
        model.fit(X_tr, y_tr, verbose=False)
        scores.append(roc_auc_score(y_val, model.predict_proba(X_val)[:, 1]))
        
    return np.mean(scores)

study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
study.optimize(objective, n_trials=30) # 데이터가 작아서 금방 끝남

best_params = study.best_params
best_params.update({'random_state': 42, 'n_jobs': -1})

print(f"Best CV AUC: {study.best_value:.4f}")

# 최종 모델 학습
base_model = xgb.XGBClassifier(**best_params)
base_model.fit(X_train, y_train)

# 평가
res = evaluate_model(base_model, X_test, y_test, 'Final Model')

# ============================================================================
# 4. 저장 (Agent 단계를 위해 필수)
# ============================================================================
# Test Set 저장 (부도 샘플이 포함된 균형 잡힌 데이터)
X_test.to_csv('X_test_final.csv', index=False)
y_test.to_csv('y_test_final.csv', index=False)

joblib.dump(base_model, 'base_model_final.pkl')
joblib.dump(res['Threshold'], 'base_model_threshold_final.pkl')
joblib.dump(list(X.columns), 'selected_features_final.pkl') # 변수명 저장

print("\n저장 완료. 다음 단계(DiCE)로 넘어가세요.")

C:\Users\ecredible\anaconda3\envs\diceml\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1. 데이터 로드
전체 데이터: (15482, 64)
전체 부도 건수: 266건 (1.72%)

2. NearMiss 언더샘플링 적용 (1:1 비율)


[I 2025-12-03 15:49:11,025] A new study created in memory with name: no-name-bae1e23b-61f8-4a14-892c-96f0f019a641


리샘플링 후 데이터 크기: (532, 64)
리샘플링 후 부도 건수: 266건
리샘플링 후 정상 건수: 266건
비율이 1:1로 조정되었습니다.
>> 'resampled_data_final.csv' 저장 완료 (시각화용)

3. XGBoost 학습 (Optuna)


[I 2025-12-03 15:49:12,053] Trial 0 finished with value: 1.0 and parameters: {'max_depth': 5, 'learning_rate': 0.28570714885887566, 'n_estimators': 380, 'min_child_weight': 5}. Best is trial 0 with value: 1.0.
[I 2025-12-03 15:49:12,412] Trial 1 finished with value: 1.0 and parameters: {'max_depth': 3, 'learning_rate': 0.055238410897498764, 'n_estimators': 76, 'min_child_weight': 7}. Best is trial 0 with value: 1.0.
[I 2025-12-03 15:49:12,690] Trial 2 finished with value: 1.0 and parameters: {'max_depth': 6, 'learning_rate': 0.21534104756085318, 'n_estimators': 59, 'min_child_weight': 7}. Best is trial 0 with value: 1.0.
[I 2025-12-03 15:49:13,150] Trial 3 finished with value: 1.0 and parameters: {'max_depth': 7, 'learning_rate': 0.07157834209670008, 'n_estimators': 132, 'min_child_weight': 2}. Best is trial 0 with value: 1.0.
[I 2025-12-03 15:49:13,850] Trial 4 finished with value: 1.0 and parameters: {'max_depth': 4, 'learning_rate': 0.16217936517334897, 'n_estimators': 244, 'min_chi

Best CV AUC: 1.0000

[Final Model] 결과
AUC: 1.0000
Optimal Threshold: 0.9767
F1 Score: 1.0000
Confusion Matrix:
[[54  0]
 [ 0 53]]
부도(1)를 맞춘 개수(TP): 53 / 53

저장 완료. 다음 단계(DiCE)로 넘어가세요.
